# Pengenalan Pola: Klasifikasi Penyakit Daun Mangga
**Anggota Kelompok:**
1. [Nama Anggota 1] - [NIM]
2. [Nama Anggota 2] - [NIM]

Proyek ini bertujuan untuk mengklasifikasikan penyakit daun mangga menggunakan ekstraksi fitur GLCM (Tekstur) dan Color Histogram (Warna) dengan algoritma klasifikasi *Support Vector Machine* (SVM).

In [ ]:
import os
import zipfile
import cv2
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from skimage.feature import graycomatrix, graycoprops
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from tqdm import tqdm

## 1. Ekstrak Dataset (File ZIP)
Bagian ini berfungsi untuk mengekstrak dataset apabila masih dalam bentuk `.zip`. Jika folder `dataset` sudah ada, proses ekstraksi akan dilewati.

In [ ]:
# Ubah nama 'dataset.zip' sesuai dengan nama file zip Anda
zip_path = 'dataset.zip'
extract_path = './dataset/'

if os.path.exists(zip_path):
    if not os.path.exists(extract_path):
        print(f"Mengekstrak {zip_path}...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_path)
        print("Ekstraksi selesai!")
    else:
        print("Folder ./dataset/ sudah ada. Ekstraksi dilewati.")
else:
    print(f"File '{zip_path}' tidak ditemukan. Jika Anda belum mengunduhnya, silakan masukkan file zip ke folder ini.")

## 2. Fungsi Ekstraksi Fitur (Handcrafted)
Kita akan mengekstrak fitur menggunakan kombinasi:
1. **GLCM (Gray-Level Co-occurrence Matrix)**: Mengambil fitur *contrast, correlation, energy,* dan *homogeneity*.
2. **Color Histogram (HSV)**: Menangkap pola distribusi warna penyakit pada daun.

In [ ]:
def extract_features(image_path):
    # 1. Baca citra dan ubah ukuran (resize) menjadi 256x256 untuk keseragaman
    img = cv2.imread(image_path)
    if img is None:
        return None
    img = cv2.resize(img, (256, 256))
    
    # 2. Ekstraksi Fitur GLCM (Tekstur)
    # GLCM bekerja pada citra keabuan (grayscale)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    
    # Menghitung matriks GLCM (jarak 1 piksel, sudut 0 derajat)
    glcm = graycomatrix(gray, distances=[1], angles=[0], levels=256, symmetric=True, normed=True)
    
    # Ekstraksi properti dari GLCM
    contrast = graycoprops(glcm, 'contrast')[0, 0]
    correlation = graycoprops(glcm, 'correlation')[0, 0]
    energy = graycoprops(glcm, 'energy')[0, 0]
    homogeneity = graycoprops(glcm, 'homogeneity')[0, 0]
    glcm_features = np.array([contrast, correlation, energy, homogeneity])
    
    # 3. Ekstraksi Fitur Color Histogram (Warna)
    # Mengubah BGR ke HSV (Hue, Saturation, Value) agar lebih peka terhadap perubahan warna penyakit
    hsv_img = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    
    # Mengambil histogram warna dengan 8 bin untuk masing-masing channel
    hist_h = cv2.calcHist([hsv_img], [0], None, [8], [0, 256]).flatten()
    hist_s = cv2.calcHist([hsv_img], [1], None, [8], [0, 256]).flatten()
    hist_v = cv2.calcHist([hsv_img], [2], None, [8], [0, 256]).flatten()
    color_features = np.concatenate((hist_h, hist_s, hist_v))
    
    # 4. Gabungkan Fitur Tekstur dan Fitur Warna
    combined_features = np.concatenate((glcm_features, color_features))
    return combined_features

## 3. Iterasi Dataset dan Persiapan Fitur
Membaca seluruh folder gambar penyakit daun, melakukan ekstraksi pada tiap gambar, dan menyimpannya dalam bentuk matriks X (fitur) dan y (label).

In [ ]:
data_dir = './dataset/'  

features = []
labels = []

# Periksa apakah direktori data ada
if os.path.exists(data_dir):
    # Mengambil semua subfolder yang ada di dalam dataset
    classes = [d for d in os.listdir(data_dir) if os.path.isdir(os.path.join(data_dir, d))]
    
    if not classes:
        print(f"Tidak ada sub-folder kelas yang ditemukan di dalam {data_dir}. Coba pastikan struktur folder zip-nya.")
        
    for cls_name in classes:
        cls_dir = os.path.join(data_dir, cls_name)
        image_files = [f for f in os.listdir(cls_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        
        # Looping dengan loading bar menggunakan tqdm
        for img_name in tqdm(image_files, desc=f"Ekstraksi fitur {cls_name}"):
            img_path = os.path.join(cls_dir, img_name)
            
            feat = extract_features(img_path)
            if feat is not None:
                features.append(feat)
                labels.append(cls_name)
else:
    print(f"Folder {data_dir} tidak ditemukan. Harap pastikan dataset sudah diekstrak.")

X = np.array(features)
y = np.array(labels)

if len(X) > 0:
    print(f"\nBerhasil memuat dataset!")
    print(f"Total Data (Baris) : {X.shape[0]} citra")
    print(f"Total Fitur (Kolom): {X.shape[1]}")
else:
    print("\nBelum ada data yang berhasil diekstrak.")

## 4. Train-Test Split
Kita membagi data menjadi 80% untuk *Training* (melatih model) dan 20% untuk *Testing* (menguji model).

In [ ]:
if len(X) > 0:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    print(f"Jumlah Data Latih: {X_train.shape[0]}")
    print(f"Jumlah Data Uji: {X_test.shape[0]}")

## 5. Pemodelan Machine Learning (SVM)
Membangun dan melatih model klasifikasi *Support Vector Machine* (SVM). Ini merupakan metode *Traditional Machine Learning*.

In [ ]:
if len(X) > 0:
    # Menggunakan Kernel Linear atau RBF (Radial Basis Function)
    svm_model = SVC(kernel='linear', C=1.0, random_state=42)
    
    print("Memulai pelatihan model SVM...")
    svm_model.fit(X_train, y_train)
    print("Pelatihan selesai!")
    
    # Evaluasi akurasi
    y_pred = svm_model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    print(f"\n>>> Akurasi Model pada Data Uji: {accuracy * 100:.2f}% <<<")

## 6. Laporan Evaluasi
Menampilkan metrik lengkap (Precision, Recall, F1-Score) dan memvisualisasikan Confusion Matrix untuk melihat kelas mana yang sering terklasifikasi dengan benar/salah.

In [ ]:
if len(X) > 0:
    print("\n--- Classification Report ---")
    print(classification_report(y_test, y_pred))
    
    # Visualisasi Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=np.unique(y), yticklabels=np.unique(y))
    plt.title('Confusion Matrix SVM - Klasifikasi Daun Mangga')
    plt.xlabel('Label Prediksi')
    plt.ylabel('Label Aktual')
    plt.xticks(rotation=45)
    plt.show()